# Chapter 3 — Score-Based Perspective: From EBMs to NCSN

This notebook follows Chapter 3 of *The Principles of Diffusion Models*. We move from the
**variational view** (Ch. 2: VAE → DDPM) to the **score-based view**: Energy-Based Models (EBMs)
→ score matching → denoising score matching (DSM) → Noise Conditional Score Networks (NCSN).

**Roadmap**
1. Energy-Based Models and why the partition function is the whole problem
2. The score function `s(x) = ∇x log p(x)` as a way to sidestep that problem
3. Vanilla (implicit) score matching + Langevin dynamics — and why it struggles
4. Denoising score matching (DSM) — a tractable, single-noise-level fix
5. NCSN — multiple noise levels + annealed Langevin dynamics
6. Comparing NCSN and DDPM side by side

All experiments use a 2D toy dataset (two moons) so we can *see* the score field and the sampling
trajectories directly — this is exactly the recommended approach before touching image data.


In [ ]:
# --- Setup (PROVIDED) ----------------------------------------------------
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# Strategy: implement TODOs one at a time, run their SELF-CHECK, then
# execute the corresponding training/sampling cell. NotImplementedError
# means that section is intentionally incomplete.
FAST_DEV_RUN = False  # True for smoke-testing during development
SM_STEPS = 150 if FAST_DEV_RUN else 3000
DSM_STEPS = 150 if FAST_DEV_RUN else 5000
NCSN_STEPS = 200 if FAST_DEV_RUN else 10000
LANG_STEPS = 80 if FAST_DEV_RUN else 1000
ALD_STEPS = 20 if FAST_DEV_RUN else 100


## 1. Energy-Based Models (EBMs)

An EBM defines a density through an **energy function** $E_\phi(x)$ that is *low* on data and
*high* everywhere else:

$$p_\phi(x) = \frac{\exp(-E_\phi(x))}{Z_\phi}, \qquad Z_\phi = \int \exp(-E_\phi(x))\,dx.$$

$Z_\phi$ (the partition function) is what makes EBMs hard: it's an integral over all of $\mathbb{R}^D$,
generally intractable, yet required for maximum-likelihood training since

$$\mathcal{L}_{\text{MLE}}(\phi) = -\,\mathbb{E}_{p_{\text{data}}}[E_\phi(x)] \;-\; \log Z_\phi.$$

The first term is easy (just lower the energy of real data). The second term — the log partition
function — needs samples from the model itself, which is expensive (MCMC, contrastive divergence, ...).

**Key escape hatch — the score.** Define the score function

$$s(x) := \nabla_x \log p(x).$$

Because $\log p_\phi(x) = -E_\phi(x) - \log Z_\phi$ and $Z_\phi$ does not depend on $x$:

$$\nabla_x \log p_\phi(x) = -\nabla_x E_\phi(x).$$

The partition function's gradient is exactly zero — **the score never sees $Z_\phi$.** This is the
whole motivation of this chapter: if we can train a network to match the *score* rather than the
*density*, we never need to touch $Z_\phi$ at all. Sampling then proceeds via **Langevin dynamics**,
which only needs the score to walk samples uphill in log-density:

$$\tilde{x}_{n+1} = \tilde{x}_n + \eta\, s(\tilde{x}_n) + \sqrt{2\eta}\,\epsilon_n, \qquad \epsilon_n \sim \mathcal{N}(0, I).$$


## 2. Score-Based Models - Training with (Explicit) Score Matching

We directly parameterize the score as a vector field $s_\phi(x) \approx \nabla_x \log p_{\text{data}}(x)$
and would like to fit it by minimizing

$$\mathcal{L}_{\text{SM}}(\phi) = \tfrac12\,\mathbb{E}_{x\sim p_{\text{data}}}\big[\|s_\phi(x) - s(x)\|_2^2\big].$$

Problem: the target $s(x)$ is unknown. **Hyvärinen's trick** (integration by parts) rewrites this,
up to an additive constant that doesn't depend on $\phi$, as a *tractable* objective that only needs
samples and the model itself:

$$\tilde{\mathcal{L}}_{\text{SM}}(\phi) = \mathbb{E}_{x\sim p_{\text{data}}}\Big[-\operatorname{Tr}\big(\nabla^2_x s_\phi(x)\big) + \tfrac12\|s_\phi(x)\|_2^2\Big].$$

The trace term is the trace of the Jacobian of the score network — i.e. it requires
second-order information (how the score itself changes with $x$). For our toy 2D data this is cheap
to compute exactly with autograd; in high dimensions it is why people later prefer *denoising* score
matching instead (Section 3).


In [ ]:
# --- Toy dataset: two moons (PROVIDED) -----------------------------------
def sample_data(n):
    x, _ = make_moons(n_samples=n, noise=0.05)
    return torch.tensor(x, dtype=torch.float32)

data = sample_data(2000)
plt.figure(figsize=(4, 4))
plt.scatter(data[:, 0], data[:, 1], s=5)
plt.title("Toy data: two moons")
plt.axis("equal")
plt.show()

# OPTIONAL TODO: estimate data.mean(0) and data.std(0), then describe how
# changing make_moons(noise=...) changes the difficulty of score estimation.


In [ ]:
# --- TODO 1: Vanilla score matching with exact divergence ---------------
# Reference: Section 2's implicit objective (Hyvarinen score matching).
# All tensors below are [B, 2] unless noted otherwise.
class ScoreNet(nn.Module):
    """Unconditional score network s_phi(x): R^2 -> R^2."""
    def __init__(self, dim=2, hidden=128):
        super().__init__()
        # TODO 1.1: Define self.net as an MLP:
        # Linear(dim,hidden) -> SiLU -> Linear(hidden,hidden) -> SiLU
        # -> Linear(hidden,dim). No output activation.
        raise NotImplementedError("TODO 1.1: unconditional score MLP")

    def forward(self, x):
        # TODO 1.2: Return predicted score using self.net.
        raise NotImplementedError("TODO 1.2: ScoreNet.forward")


def exact_trace_jacobian(score_net, x):
    """Return (trace[B], score[B,2]) using autograd."""
    # TODO 1.3: Enable gradients w.r.t. a *leaf* x. If x is already a leaf
    # you can use x.requires_grad_(True); clone/detach if needed.
    # Evaluate s=score_net(x) and initialize a length-B trace vector.
    # For each dimension j in {0,1}, use torch.autograd.grad on
    # s[:,j].sum() with create_graph=True, then add d s_j / d x_j.
    # WHY create_graph? Training must differentiate through the divergence.
    # Return (trace, s), NOT only the scalar mean trace.
    raise NotImplementedError("TODO 1.3: divergence = trace of score Jacobian")


def train_vanilla_sm(score_net, data, n_steps=3000, batch_size=256, lr=1e-3):
    # TODO 1.4: Instantiate Adam on score_net parameters.
    raise NotImplementedError("TODO 1.4: optimizer")

    for step in range(n_steps):
        # TODO 1.5: Draw a minibatch and move it to `device`.
        raise NotImplementedError("TODO 1.5: minibatch")

        # TODO 1.6: Get trace and score, and implement the objective:
        # mean over batch of [div(s_phi(x)) + 0.5 * ||s_phi(x)||_2^2].
        # Use a SUM over the coordinate dimension before the batch mean.
        # Mathematical note: divergence term is POSITIVE here.
        raise NotImplementedError("TODO 1.6: implicit SM loss")

        # TODO 1.7: Standard zero_grad -> backward -> step.
        raise NotImplementedError("TODO 1.7: train step")

        if step % 1000 == 0:
            print(f"[vanilla SM] step {step:5d}  loss {loss.item():.4f}")
    return score_net


In [ ]:
# --- SELF-CHECK 1: score network, exact Jacobian trace, training --------
# Known test field s(x) = -x has divergence -2 in 2D.
class GaussianScore(nn.Module):
    def forward(self,x): return -x
x_test = torch.randn(7, 2, device=device)
trace, s = exact_trace_jacobian(GaussianScore(), x_test.clone())
assert trace.shape == (7,) and s.shape == x_test.shape
assert torch.allclose(trace, -2 * torch.ones_like(trace), atol=1e-5)
assert torch.allclose(s, -x_test)
network_test = ScoreNet(hidden=16).to(device)
assert network_test(x_test).shape == x_test.shape
before = [p.detach().clone() for p in network_test.parameters()]
train_vanilla_sm(network_test, data, n_steps=2, batch_size=8)
assert any(not torch.equal(a,b) for a,b in zip(before, network_test.parameters()))
print("✓ SELF-CHECK 1: divergence, shape and training update")


In [ ]:
# --- Run full experiment after SELF-CHECK ---
# When TODOs 1.1–1.7 pass SELF-CHECK 1, train the model:
score_net_vanilla = ScoreNet().to(device)
score_net_vanilla = train_vanilla_sm(score_net_vanilla, data, n_steps=SM_STEPS)


In [ ]:
# --- TODO 2: Unadjusted Langevin dynamics -------------------------------
@torch.no_grad()
def langevin_sample(score_fn, n_samples=1000, dim=2, n_steps=1000, step_size=5e-4):
    # TODO 2.1: Initialize x from N(0,I) on `device`, [n_samples, dim].
    raise NotImplementedError("TODO 2.1: initial particles")
    for _ in range(n_steps):
        # TODO 2.2: Draw independent standard Gaussian noise, get score_fn(x).
        # Update x <- x + eta*s(x) + sqrt(2*eta)*noise.
        # Why is there a sqrt(2) here? Compare to Ch. 3 Langevin equation.
        raise NotImplementedError("TODO 2.2: Langevin Euler step")
    return x.cpu()

samples_vanilla = langevin_sample(score_net_vanilla,
                                 n_steps=LANG_STEPS, step_size=5e-4)
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].scatter(data[:, 0], data[:, 1], s=5, alpha=0.5)
ax[0].set_title("Data")
ax[1].scatter(samples_vanilla[:, 0], samples_vanilla[:, 1], s=5,
              alpha=0.5, color="tab:orange")
ax[1].set_title("Vanilla SM + Langevin samples")
for a in ax: a.axis("equal")
plt.show()


In [ ]:
# --- SELF-CHECK 2: Langevin shape and Gaussian-score contraction --------
# Deterministically seeded zero-noise is not needed: repeated stochastic
# simulations should have approximately correct shape and remain finite.
torch.manual_seed(45)
samples = langevin_sample(lambda x: -x, n_samples=128, n_steps=10, step_size=0.01)
assert samples.shape == (128,2) and torch.isfinite(samples).all()
print("✓ SELF-CHECK 2: Langevin sampler contract")


**What to notice:** vanilla score matching tends to produce noisy/incomplete coverage of the two
moons — the score estimate is unreliable in **low-density regions** (there's little data there to
constrain it), and Langevin dynamics needs a *good* score everywhere it walks through, including the
sparse regions between/around the modes. This is the problem NCSN is built to fix (Section 4).


## 3. Denoising Score Matching (DSM)

Instead of trying to match the score of the raw (spiky, low-coverage) data distribution, **perturb**
the data with a small amount of Gaussian noise and match the score of the *smoothed* distribution
instead. This has two benefits: (a) the noised distribution has full support everywhere, so the score
target is well-defined even far from the original data, and (b) the regression target becomes
**closed-form**, so no second derivatives are needed at all.

Let $\tilde{x} = x + \sigma \epsilon,\ \epsilon\sim\mathcal N(0,I)$, so the perturbation kernel is
$p_\sigma(\tilde x \mid x) = \mathcal N(\tilde x; x, \sigma^2 I)$. Its score has a closed form:

$$\nabla_{\tilde x}\log p_\sigma(\tilde x \mid x) = \frac{x - \tilde x}{\sigma^2}.$$

Vincent's theorem says that matching this *conditional* score is (up to a constant) equivalent to
matching the score of the true marginal $p_\sigma(\tilde x)$. The **DSM objective**:

$$\mathcal{L}_{\text{DSM}}(\phi;\sigma) = \tfrac12\,\mathbb{E}_{x,\epsilon}\Big[\big\|\,s_\phi(x+\sigma\epsilon;\sigma) + \tfrac{\epsilon}{\sigma}\,\big\|_2^2\Big].$$

This is just a **regression problem**: predict $-\epsilon/\sigma$ from the noisy input. No Jacobian
trace, no partition function — purely a supervised MSE loss.


In [ ]:
# --- TODO 3: single-noise-level denoising score matching -----------------
SIGMA_DSM = 0.05

def train_dsm(score_net, data, sigma, n_steps=5000, batch_size=256, lr=1e-3):
    # TODO 3.1: Create Adam optimizer for the score network.
    raise NotImplementedError("TODO 3.1: DSM optimizer")
    for step in range(n_steps):
        # TODO 3.2: Sample a data minibatch [B,2] and Gaussian eps [B,2].
        raise NotImplementedError("TODO 3.2: sample training pairs")
        # TODO 3.3: Corrupt x with sigma*eps and compute the score target
        # ∇_{x_noisy} log p(x_noisy|x) = -eps/sigma.
        raise NotImplementedError("TODO 3.3: DSM noisy input and target")
        # TODO 3.4: Predict score_net(x_noisy), minimize mean per-example
        # squared L2 error. No input Jacobians or higher derivatives here.
        raise NotImplementedError("TODO 3.4: DSM objective")
        # TODO 3.5: Optimize using zero_grad(), backward(), step().
        raise NotImplementedError("TODO 3.5: DSM update")
        if step % 1000 == 0:
            print(f"[DSM sigma={sigma}] step {step:5d}  loss {loss.item():.4f}")
    return score_net


In [ ]:
# --- SELF-CHECK 3: DSM applies optimizer and computes finite gradient ----
test_model=ScoreNet(hidden=16).to(device)
prev=[p.detach().clone() for p in test_model.parameters()]
assert train_dsm(test_model, data, sigma=0.2, n_steps=2, batch_size=8) is test_model
assert any(not torch.equal(a,b) for a,b in zip(prev,test_model.parameters()))
assert all(torch.isfinite(p).all() for p in test_model.parameters())
print("✓ SELF-CHECK 3: DSM training update")


In [ ]:
# --- Run full experiment after SELF-CHECK ---
score_net_dsm = ScoreNet().to(device)
score_net_dsm = train_dsm(score_net_dsm, data, SIGMA_DSM,
                          n_steps=DSM_STEPS)


In [ ]:
# --- Sample with Langevin dynamics using the DSM score --------------------
samples_dsm = langevin_sample(score_net_dsm, n_steps=LANG_STEPS, step_size=0.01)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].scatter(data[:, 0], data[:, 1], s=5, alpha=0.5)
ax[0].set_title("Data")
ax[1].scatter(samples_vanilla[:, 0], samples_vanilla[:, 1], s=5, alpha=0.5, color="tab:orange")
ax[1].set_title("Vanilla SM")
ax[2].scatter(samples_dsm[:, 0], samples_dsm[:, 1], s=5, alpha=0.5, color="tab:green")
ax[2].set_title(f"DSM (sigma={SIGMA_DSM})")
for a in ax: a.axis("equal")
plt.show()
# OPTIONAL TODO: sample with sigma=0.02 and 0.2; compare the two distributions.


**What to notice:** DSM is more *stable* to train (pure regression, no second derivatives) and
usually gives cleaner samples than vanilla SM. But there's still a fundamental tension baked into the
single fixed sigma:
- **too small** sigma → score estimate is accurate near the data, but Langevin still struggles to
  move *between* the two moons (vanishing gradients in the gap between modes).
- **too large** sigma → easy to sample from (no gaps left), but the noised distribution has lost
  the fine structure of the two moons, so samples are blurry/over-smoothed.

One noise level can't have it both ways. NCSN's fix: use **many** noise levels at once.


## 4. Noise Conditional Score Networks (NCSN)

Song & Ermon (2019) train a **single, noise-conditional** score network $s_\phi(x, \sigma)$ across a
whole ladder of noise levels $0 < \sigma_1 < \sigma_2 < \dots < \sigma_L$ ($\sigma_1$ small enough to
preserve detail, $\sigma_L$ large enough to smooth out the modes and make the space easy to explore).

**Training objective** — just DSM, summed over all levels with a per-level weight $\lambda(\sigma_i)$:

$$\mathcal{L}_{\text{NCSN}}(\phi) = \sum_{i=1}^L \lambda(\sigma_i)\, \mathcal{L}_{\text{DSM}}(\phi;\sigma_i),
\qquad
\mathcal{L}_{\text{DSM}}(\phi;\sigma) = \tfrac12\,\mathbb{E}\Big[\big\|s_\phi(\tilde x,\sigma) - \tfrac{x-\tilde x}{\sigma^2}\big\|_2^2\Big].$$

A common, convenient weighting is $\lambda(\sigma_i) = \sigma_i^2$, which puts all noise levels on a
comparable loss scale.

**Sampling — Annealed Langevin Dynamics.** Start from pure noise at the *largest* scale $\sigma_L$,
run a few Langevin steps there, then use that result to *initialize* Langevin at the next smaller
scale $\sigma_{L-1}$, and so on down to $\sigma_1 \approx 0$. Coarse structure is resolved first
(easy, smooth landscape); fine detail is resolved last (hard, but now well-initialized):

```
x ~ N(0, sigma_L^2 I)
for l = L, L-1, ..., 1:
    run K steps of Langevin dynamics at noise level sigma_l, using s_phi(., sigma_l)
    (each level's output initializes the next, smaller level)
return x   # approx. sample from p_data
```

The step size is typically scaled with the noise level, $\eta_\ell = \delta\cdot(\sigma_\ell/\sigma_1)^2$,
so early (coarse, high-noise) steps move further and later (fine, low-noise) steps move more gently.


In [ ]:
# --- TODO 4: Noise-conditional score network ----------------------------
class NCSN(nn.Module):
    """s_phi(x,sigma) has shape [B,2] and accepts scalar or [B] sigma."""
    def __init__(self, dim=2, hidden=128):
        super().__init__()
        # TODO 4.1: Build an MLP whose input is [x1,x2,log(sigma)]
        # (input width dim+1, hidden width hidden, output width dim).
        # Use 2 hidden layers, with SiLU nonlinearity.
        raise NotImplementedError("TODO 4.1: noise-conditional MLP")

    def forward(self, x, sigma):
        # TODO 4.2: If sigma is 0-D, broadcast to [B]; otherwise expect [B].
        # Compute log(sigma) as [B,1], concatenate with x along dim=1.
        # Return the MLP output, a [B,dim] tensor.
        raise NotImplementedError("TODO 4.2: log-sigma conditioning")


In [ ]:
# --- SELF-CHECK 4: per-example sigma vs scalar sigma --------------------
net=NCSN(hidden=16).to(device)
x=torch.randn(4,2,device=device)
scalar=torch.tensor(0.1,device=device)
a=net(x,scalar)
b=net(x,torch.full((4,),0.1,device=device))
assert a.shape==(4,2) and b.shape==(4,2)
torch.testing.assert_close(a,b)
assert torch.isfinite(a).all()
print("✓ SELF-CHECK 4: NCSN broadcasting and output shapes")


In [ ]:
# --- TODO 5: geometric noise schedule -----------------------------------
L = 10
sigma_min, sigma_max = 0.02, 0.2
# TODO 5.1: Build `sigmas` as L log-spaced numbers from sigma_max
# down to sigma_min, on the chosen device. Try torch.linspace on the
# logarithms followed by torch.exp. Its order should be LARGE -> SMALL.
raise NotImplementedError("TODO 5.1: geometric schedule")
print("noise levels (sigma_L -> sigma_1):", [round(s.item(), 3) for s in sigmas])


In [ ]:
# --- SELF-CHECK 5: schedule boundaries, monotonicity and ratios ---------
assert sigmas.shape == (L,) and sigmas.device.type == device
assert torch.all(sigmas[:-1] > sigmas[1:])
assert torch.isclose(sigmas[0],torch.tensor(sigma_max,device=device),atol=1e-5)
assert torch.isclose(sigmas[-1],torch.tensor(sigma_min,device=device),atol=1e-5)
ratios=sigmas[1:]/sigmas[:-1]
assert torch.allclose(ratios,ratios[0].expand_as(ratios),atol=1e-5)
print("✓ SELF-CHECK 5: geometric noise ladder")


In [ ]:
# --- TODO 6: multi-noise-level DSM (NCSN) -------------------------------
def train_ncsn(model, data, sigmas, n_steps=5000, batch_size=256, lr=1e-3):
    # TODO 6.1: Initialize Adam on model.parameters(); determine L.
    raise NotImplementedError("TODO 6.1: optimizer and L")
    for step in range(n_steps):
        # TODO 6.2: Choose random x0 minibatch; choose noise LEVEL INDEX
        # independently for each example; gather sigma_batch [B].
        raise NotImplementedError("TODO 6.2: sample data and noise levels")
        # TODO 6.3: Generate x_noisy=x0+sigma_batch[:,None]*eps;
        # conditional score target = -eps/sigma_batch[:,None].
        raise NotImplementedError("TODO 6.3: sigma-conditioned DSM targets")
        # TODO 6.4: Predict model(x_noisy,sigma_batch) and minimize
        # mean_B [sigma_batch^2 * sum_coordinates (pred-target)^2].
        # The sigma^2 weighting is crucial for balancing scales.
        raise NotImplementedError("TODO 6.4: weighted multi-noise DSM loss")
        # TODO 6.5: Standard optimizer gradient step.
        raise NotImplementedError("TODO 6.5: optimize")
        if step % 1000 == 0:
            print(f"[NCSN] step {step:5d}  loss {loss.item():.4f}")
    return model


In [ ]:
# --- SELF-CHECK 6: multi-scale objective updates weights ----------------
test_model=NCSN(hidden=16).to(device)
before=[p.detach().clone() for p in test_model.parameters()]
train_ncsn(test_model,data,sigmas,n_steps=2,batch_size=16)
assert any(not torch.equal(a,b) for a,b in zip(before,test_model.parameters()))
assert all(torch.isfinite(p).all() for p in test_model.parameters())
print("✓ SELF-CHECK 6: NCSN training")


In [ ]:
# --- Run full experiment after SELF-CHECK ---
ncsn = NCSN().to(device)
ncsn = train_ncsn(ncsn, data, sigmas, n_steps=NCSN_STEPS)


In [ ]:
# --- TODO 7: annealed Langevin dynamics (ALD) ----------------------------
@torch.no_grad()
def annealed_langevin_sample(model, sigmas, n_samples=1000, dim=2,
                             n_steps_per_level=100, delta=5e-4):
    # TODO 7.1: Initialize from N(0, sigma_L^2 I) where sigmas[0]
    # is the LARGEST noise scale, then remember sigma_1=sigmas[-1].
    raise NotImplementedError("TODO 7.1: high-noise initialization")
    for sigma in sigmas:
        # TODO 7.2: Scale step size eta = delta*(sigma/sigma_1)^2.
        raise NotImplementedError("TODO 7.2: level-dependent step size")
        for _ in range(n_steps_per_level):
            # TODO 7.3: Evaluate model(x,sigma_batch) at the CURRENT level,
            # adding fresh Gaussian noise and taking a Langevin step:
            # x <- x + eta*score + sqrt(2*eta)*noise.
            # sigma_batch should have shape [n_samples].
            raise NotImplementedError("TODO 7.3: Langevin inner loop")
    return x.cpu()

samples_ncsn = annealed_langevin_sample(ncsn, sigmas,
                                       n_steps_per_level=ALD_STEPS)


In [ ]:
# --- SELF-CHECK 7: annealed sampler contracts ---------------------------
# Testing shape/finite outputs with a simple oracle, no training required.
class GaussianConditional(nn.Module):
    def forward(self,x,sigma): return -x/(1+sigma[:,None]**2)
s=annealed_langevin_sample(GaussianConditional(),
    torch.tensor([0.5,0.2,0.05],device=device),n_samples=64,
    n_steps_per_level=3,delta=1e-4)
assert s.shape==(64,2) and torch.isfinite(s).all()
print("✓ SELF-CHECK 7: annealed Langevin sampling")


In [ ]:
# --- Compare: data vs. single-scale DSM vs. NCSN (annealed) ---------------
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].scatter(data[:, 0], data[:, 1], s=5, alpha=0.5)
ax[0].set_title("Data")
ax[1].scatter(samples_dsm[:, 0], samples_dsm[:, 1], s=5, alpha=0.5, color="tab:green")
ax[1].set_title("Single-scale DSM")
ax[2].scatter(samples_ncsn[:, 0], samples_ncsn[:, 1], s=5, alpha=0.5, color="tab:red")
ax[2].set_title("NCSN (annealed Langevin)")
for a in ax: a.axis("equal")
plt.show()

# OPTIONAL TODO: plot original vs DSM vs NCSN under matched axis limits.


**What to notice:** the annealed sampler should cover both moons more faithfully and with
sharper structure than the single-scale DSM sampler — the high-noise levels let it explore and find
both modes (no vanishing-gradient gap to get stuck in), while the low-noise levels at the end recover
fine detail. This coverage-vs-detail trade-off, solved by annealing, is precisely the point of NCSN.


In [ ]:
# --- Visualize the learned score field at a few noise levels --------------
def plot_score_field(model, sigma, ax, xlim=(-2, 3), ylim=(-2, 2), grid=20):
    xs = torch.linspace(*xlim, grid)
    ys = torch.linspace(*ylim, grid)
    X, Y = torch.meshgrid(xs, ys, indexing="xy")
    pts = torch.stack([X.flatten(), Y.flatten()], dim=1).to(device)
    with torch.no_grad():
        s = model(pts, torch.tensor(sigma, device=device).expand(pts.shape[0])).cpu()
    ax.quiver(pts[:, 0], pts[:, 1], s[:, 0], s[:, 1], color="gray", alpha=0.6)
    ax.set_title(f"score field, sigma={sigma:.2f}")
    ax.axis("equal")

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, sigma in zip(ax, [sigmas[0].item(), sigmas[L // 2].item(), sigmas[-1].item()]):
    plot_score_field(ncsn, sigma, a)
plt.tight_layout()
plt.show()

# OPTIONAL TODO: plot unit-normalized arrows as well as raw magnitudes.


Notice how the field is smoother/coarser (points more uniformly toward the overall mass) at
large sigma, and sharper/more structured (clearly outlining the two moons) at small sigma — this
is the visual signature of what annealing exploits.


## 5. NCSN vs. DDPM — Side by Side

Both models perturb data with noise and learn to undo it, and it turns out their optimal targets are
**exact rescalings of each other** (via Tweedie's formula):

$$s^*(x_\sigma, \sigma) = -\frac{1}{\sigma}\,\mathbb{E}[\epsilon \mid x_\sigma] = -\frac{1}{\sigma}\,\epsilon^*(x_\sigma,\sigma).$$

So a perfectly trained DDPM noise-predictor and a perfectly trained NCSN score-predictor encode the
same information, just reparameterized.

| | **NCSN** | **DDPM** |
|---|---|---|
| Forward step | $x_{i+1}=x_i+\sqrt{\sigma_{i+1}^2-\sigma_i^2}\,\epsilon$ | $x_{i+1}=\sqrt{1-\beta_i}\,x_i+\sqrt{\beta_i}\,\epsilon$ |
| $x_i \mid x$ | $x_i = x + \sigma_i \epsilon$ | $x_i = \bar\alpha_i x + \sqrt{1-\bar\alpha_i^2}\,\epsilon$ |
| Prior | $\mathcal N(0,\sigma_L^2 I)$ | $\mathcal N(0, I)$ |
| Loss target | score $s_\phi(x_i,\sigma_i)\approx -\epsilon/\sigma_i$ | noise $\epsilon_\phi(x_i,i)\approx\epsilon$ |
| Sampling | Langevin at each level, annealed | ancestral sampling through $p_\phi(x_{i-1}\mid x_i)$ |

**Shared bottleneck.** Both need many discrete levels/steps (dense noise schedule) to work well,
which is why sampling from either is slow (hundreds–thousands of network evaluations). This shared
limitation is exactly what motivates the continuous-time **Score SDE** framework of Chapter 4, which
unifies NCSN and DDPM as two discretizations of the same underlying stochastic process.


## 6. Exercises / Checkpoint

1. Re-run vanilla score matching with a lower Langevin step size / more steps — does coverage of
   both moons improve, or does it just get slower without fixing the underlying issue?
2. Try DSM with a much larger sigma (e.g. 1.0) and a much smaller one (e.g. 0.05). Confirm the
   trade-off described above: large sigma -> easy sampling but blurred structure; small sigma ->
   sharp but Langevin struggles to jump between moons.
3. Reduce `L` (number of noise levels) in NCSN to 2 or 3 — how much does annealed sample quality
   degrade? This should give you intuition for why NCSN eventually motivates a *continuum* of noise
   levels (Chapter 4's Score SDE).
4. In your own words (before peeking back at Section 5's table): why must NCSN and DDPM samplers
   both be slow, given that they're mathematically equivalent at the level of the optimal score?

**Checkpoint:** you should be able to explain why single-noise-level score matching fails on
multimodal data, and why annealing across many noise levels fixes it.
